In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load data
df = pd.read_csv("C:\\Users\\Renu Sharma\\Downloads\\Nat_Gas.csv")
df['Dates'] = pd.to_datetime(df['Dates'])
df = df.sort_values('Dates').reset_index(drop=True)

C:\Users\Renu Sharma\AppData\Local\Temp\ipykernel_11180\145694648.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dates'] = pd.to_datetime(df['Dates'])


In [8]:
# Time in years from start
df['t'] = (df['Dates'] - df['Dates'].min()).dt.days / 365.25

# Linear trend
slope, intercept = np.polyfit(df['t'], df['Prices'], 1)

# Seasonal adjustments (monthly averages of residuals)
df['month'] = df['Dates'].dt.month
df['residual'] = df['Prices'] - (intercept + slope * df['t'])
seasonal = df.groupby('month')['residual'].mean()

# Price estimator function
def estimate_price(date_str):
    d = pd.Timestamp(date_str)
    t = (d - df['Dates'].min()).dt.days / 365.25 if hasattr((d - df['Dates'].min()), 'dt') \
        else (d - df['Dates'].min()).days / 365.25
    price = intercept + slope * t + seasonal[d.month]
    return round(price, 2)

# Test it
print(estimate_price('2024-06-30'))   # historical date
print(estimate_price('2025-06-30'))   # future extrapolation

11.4
11.87
